In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import json
import os

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

COLORS = {
    'Doc': '#4C72B0', 'Img': '#DD8452', 'Movie': '#55A868',
    'Rec': '#C44E52', 'BGM': '#8172B3'
}
DPI = 300

# Load ablation data
with open('../publication/paper/_asf_ablation_results.json', 'r') as f:
    asf_data = json.load(f)

with open('../publication/paper/_rerank_ablation_results.json', 'r') as f:
    rerank_data = json.load(f)

with open('../publication/paper/_langgraph_ablation_results.json', 'r') as f:
    langgraph_data = json.load(f)

In [ ]:
# fig04_asf_ablation.png - ASF On/Off comparison
domains = ['Doc', 'Img', 'Movie', 'Rec']
hit_off  = [0.0,  1.0,  0.95, 1.0]
hit_on   = [0.0,  1.0,  0.95, 1.0]
lat_off  = [77.9, 38.1, 51.1, 45.4]
lat_on   = [84.5, 38.0, 63.3, 44.0]

x = np.arange(len(domains))
width = 0.35

fig, ax1 = plt.subplots(figsize=(12, 7))

bars1 = ax1.bar(x - width/2, hit_off, width, label='ASF Off', color='#4C72B0', alpha=0.85)
bars2 = ax1.bar(x + width/2, hit_on,  width, label='ASF On',  color='#DD8452', alpha=0.85)

ax1.set_xlabel('도메인', fontsize=13)
ax1.set_ylabel('Hit@5', fontsize=13, color='black')
ax1.set_ylim(0, 1.25)
ax1.set_xticks(x)
ax1.set_xticklabels(domains, fontsize=12)
ax1.tick_params(axis='y', labelcolor='black')

for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}',
             ha='center', va='bottom', fontsize=10, color='#4C72B0')
for bar in bars2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}',
             ha='center', va='bottom', fontsize=10, color='#DD8452')

ax2 = ax1.twinx()
ax2.plot(x - width/2, lat_off, 'o--', color='#4C72B0', linewidth=1.8,
         markersize=7, label='Latency p95 Off')
ax2.plot(x + width/2, lat_on,  's--', color='#DD8452', linewidth=1.8,
         markersize=7, label='Latency p95 On')
ax2.set_ylabel('Latency p95 (ms)', fontsize=13, color='dimgray')
ax2.tick_params(axis='y', labelcolor='dimgray')
ax2.set_ylim(0, 150)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

plt.title('ASF 필터 Ablation (Hit@5 & Latency)', fontsize=15, fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig('fig04_asf_ablation.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('Saved fig04_asf_ablation.png')

In [ ]:
# fig04_reranker_ablation.png - Reranker On/Off comparison
domains_rr = ['Movie', 'Rec']
no_hit   = [0.967, 1.0]
with_hit = [0.333, 0.2]
no_lat   = [84.9,  69.0]
with_lat = [550.9, 603.8]

fig, axes = plt.subplots(1, 2, figsize=(12, 7))

# --- Hit@5 subplot ---
ax = axes[0]
x2 = np.arange(len(domains_rr))
b1 = ax.bar(x2 - 0.2, no_hit,   0.35, label='Reranker Off', color='#55A868', alpha=0.85)
b2 = ax.bar(x2 + 0.2, with_hit, 0.35, label='Reranker On',  color='#C44E52', alpha=0.85)

for bar in b1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
            ha='center', va='bottom', fontsize=10)
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
            ha='center', va='bottom', fontsize=10, color='#C44E52', fontweight='bold')

# Annotation arrows for degradation
for i, (d, nv, wv) in enumerate(zip(domains_rr, no_hit, with_hit)):
    drop = nv - wv
    ax.annotate('', xy=(i + 0.2, wv + 0.04), xytext=(i - 0.2, nv - 0.04),
                arrowprops=dict(arrowstyle='->', color='red', lw=2.0))
    ax.text(i + 0.25, (nv + wv) / 2, f'-{drop:.3f}',
            color='red', fontsize=10, fontweight='bold')

ax.set_xticks(x2)
ax.set_xticklabels(domains_rr, fontsize=12)
ax.set_ylabel('Hit@5', fontsize=13)
ax.set_ylim(0, 1.3)
ax.set_title('Hit@5 비교', fontsize=13)
ax.legend(fontsize=11)

# --- Latency subplot ---
ax2 = axes[1]
b3 = ax2.bar(x2 - 0.2, no_lat,   0.35, label='Reranker Off', color='#55A868', alpha=0.85)
b4 = ax2.bar(x2 + 0.2, with_lat, 0.35, label='Reranker On',  color='#C44E52', alpha=0.85)

for bar in b3:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 5, f'{h:.1f}',
             ha='center', va='bottom', fontsize=10)
for bar in b4:
    h = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2, h + 5, f'{h:.1f}',
             ha='center', va='bottom', fontsize=10, color='#C44E52', fontweight='bold')

# Annotation for latency explosion
for i, (nv, wv) in enumerate(zip(no_lat, with_lat)):
    ratio = wv / nv
    ax2.text(i + 0.28, wv * 0.55, f'x{ratio:.1f}',
             color='red', fontsize=11, fontweight='bold')

ax2.set_xticks(x2)
ax2.set_xticklabels(domains_rr, fontsize=12)
ax2.set_ylabel('Latency p95 (ms)', fontsize=13)
ax2.set_title('Latency p95 비교', fontsize=13)
ax2.legend(fontsize=11)

fig.suptitle('Reranker Ablation: Cross-Encoder 역효과', fontsize=15, fontweight='bold', y=1.02)
fig.tight_layout()
plt.savefig('fig04_reranker_ablation.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('Saved fig04_reranker_ablation.png')

In [ ]:
# fig04_langgraph_ablation.png - LangGraph Query Rewriting
domains_lg = ['Movie', 'Rec']
no_rw   = [0.867, 0.2]
with_rw = [0.867, 0.2]

x3 = np.arange(len(domains_lg))
width_lg = 0.35

fig, ax = plt.subplots(figsize=(10, 6))

b1 = ax.bar(x3 - width_lg/2, no_rw,   width_lg, label='Rewrite 없음', color='#4C72B0', alpha=0.85)
b2 = ax.bar(x3 + width_lg/2, with_rw, width_lg, label='Rewrite 적용', color='#8172B3', alpha=0.85)

for bar in b1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
            ha='center', va='bottom', fontsize=11)
for bar in b2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.015, f'{h:.3f}',
            ha='center', va='bottom', fontsize=11)

ax.set_xticks(x3)
ax.set_xticklabels(domains_lg, fontsize=13)
ax.set_ylabel('Hit@5', fontsize=13)
ax.set_ylim(0, 1.2)
ax.legend(fontsize=12)

# Annotation: rewrite not fired
ax.text(0.5, 1.08,
        'z > 1.0 → rewrite 미발동 (0회)\n두 조건 모두 결과 동일',
        transform=ax.transAxes, ha='center', va='center',
        fontsize=12, color='#555555',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFFDE7', edgecolor='#BDBDBD', alpha=0.9))

plt.title('LangGraph Query Rewriting Ablation', fontsize=15, fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig('fig04_langgraph_ablation.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('Saved fig04_langgraph_ablation.png')

In [ ]:
# fig04_ablation_summary.png - Summary table as image
fig, ax = plt.subplots(figsize=(14, 5))
ax.set_axis_off()

col_labels = ['구성 요소', '적용 도메인', '정확도 영향', '지연시간 영향', '결론']
row_data = [
    ['ASF 필터',    'Doc / Img / Movie / Rec', '변화 없음 (0.0~1.0)',  '+6~12ms 미만',       '구조적 필수 (오답 방지)'],
    ['Lexical 검색', 'Movie / Rec',            '보완적 역할',           '무시 가능',          '하이브리드 유지 권장'],
    ['Reranker',    'Movie / Rec',             '심각 저하 (↓63~80%)', '폭증 (+466~534ms)', '제거 결정 (역효과 확인)'],
    ['LangGraph RW', 'Movie / Rec',            '변화 없음',             '미측정',             '발동 조건 재설계 필요'],
]

# Cell colors
cell_colors = []
for row in row_data:
    row_colors = ['#F5F5F5'] * len(col_labels)
    acc_text = row[2]
    lat_text = row[3]
    conc_text = row[4]

    # accuracy column (index 2)
    if '저하' in acc_text or '↓' in acc_text:
        row_colors[2] = '#FFCDD2'
    elif '변화 없음' in acc_text:
        row_colors[2] = '#E0E0E0'
    else:
        row_colors[2] = '#C8E6C9'

    # latency column (index 3)
    if '폭증' in lat_text or '+' in lat_text:
        row_colors[3] = '#FFCDD2'
    elif '무시' in lat_text or '미측정' in lat_text:
        row_colors[3] = '#E0E0E0'
    else:
        row_colors[3] = '#C8E6C9'

    # conclusion column (index 4)
    if '제거' in conc_text or '역효과' in conc_text:
        row_colors[4] = '#FFCDD2'
    elif '재설계' in conc_text:
        row_colors[4] = '#FFF9C4'
    else:
        row_colors[4] = '#C8E6C9'

    cell_colors.append(row_colors)

table = ax.table(
    cellText=row_data,
    colLabels=col_labels,
    cellColours=cell_colors,
    colColours=['#1565C0'] * len(col_labels),
    loc='center',
    cellLoc='center'
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.0, 2.2)

# Header text color white
for j in range(len(col_labels)):
    table[0, j].get_text().set_color('white')
    table[0, j].get_text().set_fontweight('bold')

# Row label bold
for i in range(1, len(row_data) + 1):
    table[i, 0].get_text().set_fontweight('bold')

plt.title('Ablation Study 종합 요약', fontsize=16, fontweight='bold', pad=20)
fig.tight_layout()
plt.savefig('fig04_ablation_summary.png', dpi=DPI, bbox_inches='tight')
plt.close()
print('Saved fig04_ablation_summary.png')